<a href="https://colab.research.google.com/github/workatustadarshajay/Docs_Code_Testing/blob/main/Image_MASKING_Presidio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install \
    presidio-analyzer \
    presidio-anonymizer \
    spacy \
    pytesseract \
    pillow \
    opencv-python

In [ ]:
!python -m spacy download en_core_web_lg

In [ ]:
!apt-get -qq update
!apt-get -qq install tesseract-ocr

In [ ]:
import os
import re
import json
import logging
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import pytesseract
import cv2
import numpy as np

from PIL import Image, ImageDraw, ImageFont

try:
    from google.colab import files
    IN_COLAB = True
except:
    IN_COLAB = False

from presidio_analyzer import (
    AnalyzerEngine,
    RecognizerRegistry,
    PatternRecognizer,
    Pattern,
)

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger("image-masking")

PII_ENTITIES = [
    "PERSON",
    "EMAIL_ADDRESS",
    "PHONE_NUMBER",
    "CREDIT_CARD",
    "US_SSN",
    "IBAN_CODE",
    "IP_ADDRESS",
    "LOCATION",
    "DATE_TIME",
    "URL",
]

In [ ]:
registry = RecognizerRegistry()
registry.load_predefined_recognizers()

analyzer = AnalyzerEngine(registry=registry)

print("Presidio initialized successfully.")

In [ ]:
member_id_pattern = Pattern(
    name="member_id",
    regex=r"\b[A-Z]{2,5}[- ]?\d{6,12}\b",
    score=0.70,
)

member_id_recognizer = PatternRecognizer(
    supported_entity="MEMBER_ID",
    patterns=[member_id_pattern],
)

policy_pattern = Pattern(
    name="policy_number",
    regex=r"\b(?:POL|POLICY)[- ]?[A-Z0-9]{5,15}\b",
    score=0.75,
)

policy_recognizer = PatternRecognizer(
    supported_entity="POLICY_NUMBER",
    patterns=[policy_pattern],
)

mrn_pattern = Pattern(
    name="medical_record_number",
    regex=r"\b(?:MRN|MR)[- ]?\d{5,15}\b",
    score=0.80,
)

mrn_recognizer = PatternRecognizer(
    supported_entity="MEDICAL_RECORD_NUMBER",
    patterns=[mrn_pattern],
)

authorization_pattern = Pattern(
    name="authorization_number",
    regex=r"\b(?:AUTH|AUTHORIZATION)[- ]?[A-Z0-9]{5,20}\b",
    score=0.80,
)

authorization_recognizer = PatternRecognizer(
    supported_entity="AUTHORIZATION_NUMBER",
    patterns=[authorization_pattern],
)

registry.add_recognizer(member_id_recognizer)
registry.add_recognizer(policy_recognizer)
registry.add_recognizer(mrn_recognizer)
registry.add_recognizer(authorization_recognizer)

analyzer = AnalyzerEngine(registry=registry)

print("Custom healthcare recognizers added.")

In [ ]:
if IN_COLAB:
    uploaded = files.upload()
    image_name = next(iter(uploaded))
    print(f"Uploaded: {image_name}")
else:
    image_name = "test_image.png"
    print(f"Using local image: {image_name}")

In [ ]:
@dataclass
class TextRegion:
    text: str
    x: int
    y: int
    w: int
    h: int
    confidence: float

In [ ]:
def extract_text_regions(image_path: str) -> Tuple[str, List[TextRegion]]:
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Image not found: {image_path}")
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    data = pytesseract.image_to_data(
        gray,
        output_type=pytesseract.Output.DICT
    )
    
    full_text = " ".join(data["text"])
    regions = []
    
    for i, text in enumerate(data["text"]):
        if text.strip():
            confidence = int(data["confidence"][i])
            if confidence > 0:
                regions.append(
                    TextRegion(
                        text=text,
                        x=data["left"][i],
                        y=data["top"][i],
                        w=data["width"][i],
                        h=data["height"][i],
                        confidence=confidence / 100.0,
                    )
                )
    
    return full_text, regions

In [ ]:
full_text, regions = extract_text_regions(image_name)

print(f"Extracted text: {len(full_text)} characters")
print(f"Found {len(regions)} text regions")
print(f"\nSample text:\n{full_text[:200]}...")

In [ ]:
def detect_pii(text: str):
    results = analyzer.analyze(
        text=text,
        language="en",
        score_threshold=0.35,
    )
    return results

In [ ]:
def region_overlaps_entity(region: TextRegion, entity_start: int, entity_end: int, text: str) -> bool:
    region_start = text.find(region.text)
    if region_start == -1:
        return False
    region_end = region_start + len(region.text)
    
    return max(region_start, entity_start) < min(region_end, entity_end)

In [ ]:
def find_regions_to_mask(text: str, regions: List[TextRegion]) -> List[Dict]:
    pii_results = detect_pii(text)
    
    print(f"\nFound {len(pii_results)} PII entities")
    
    mask_regions = []
    
    for result in pii_results:
        entity_value = text[result.start:result.end]
        
        print(
            f"  {result.entity_type}: {entity_value!r} "
            f"score={result.score:.2f}"
        )
        
        for region in regions:
            if region_overlaps_entity(region, result.start, result.end, text):
                mask_regions.append({
                    "x": region.x,
                    "y": region.y,
                    "w": region.w,
                    "h": region.h,
                    "entity": result.entity_type,
                    "score": result.score,
                })
    
    return mask_regions

In [ ]:
mask_regions = find_regions_to_mask(full_text, regions)
print(f"\nTotal regions to mask: {len(mask_regions)}")

In [ ]:
def create_masked_image(image_path: str, output_path: str, mask_regions: List[Dict], padding: int = 5):
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Image not found: {image_path}")
    
    for region in mask_regions:
        x = max(0, region["x"] - padding)
        y = max(0, region["y"] - padding)
        x2 = min(img.shape[1], region["x"] + region["w"] + padding)
        y2 = min(img.shape[0], region["y"] + region["h"] + padding)
        
        cv2.rectangle(
            img,
            (x, y),
            (x2, y2),
            (0, 0, 0),
            -1
        )
    
    cv2.imwrite(output_path, img)
    
    return output_path

In [ ]:
output_image = "masked_image.png"

create_masked_image(
    image_path=image_name,
    output_path=output_image,
    mask_regions=mask_regions,
)

print(f"Created: {output_image}")

In [ ]:
def compare_images(original_path: str, masked_path: str):
    orig = cv2.imread(original_path)
    masked = cv2.imread(masked_path)
    
    if orig is None or masked is None:
        print("Error reading images")
        return
    
    orig_gray = cv2.cvtColor(orig, cv2.COLOR_BGR2GRAY)
    masked_gray = cv2.cvtColor(masked, cv2.COLOR_BGR2GRAY)
    
    diff = cv2.absdiff(orig_gray, masked_gray)
    masked_pixels = np.count_nonzero(diff)
    total_pixels = orig_gray.size
    
    coverage = (masked_pixels / total_pixels) * 100
    
    print(f"\nMasking coverage: {coverage:.2f}% of image modified")
    print(f"Pixels changed: {masked_pixels:,} / {total_pixels:,}")
    
    return coverage

In [ ]:
compare_images(image_name, output_image)

In [ ]:
def verify_masking(original_path: str, masked_path: str) -> bool:
    orig_text, _ = extract_text_regions(original_path)
    masked_text, _ = extract_text_regions(masked_path)
    
    pii_results = detect_pii(orig_text)
    
    remaining_count = 0
    
    for result in pii_results:
        value = orig_text[result.start:result.end]
        
        if value.lower() in masked_text.lower():
            remaining_count += 1
            print(f"⚠️  Potentially remaining: {value!r}")
    
    if remaining_count == 0:
        print("✅ MASKING VERIFICATION PASSED")
        return True
    else:
        print(f"❌ {remaining_count} PII value(s) may remain visible")
        return False

In [ ]:
verify_masking(image_name, output_image)

In [ ]:
if IN_COLAB:
    files.download(output_image)
else:
    print(f"Masked image saved to: {output_image}")